In [1]:
"""
Author: Shoshana Abikzer, Rebecca Gou

Goal:
Compute annual mean NDVI (MODIS) and mean nighttime lights (VIIRS) for Chengdu ADM3 units
using the Google Earth Engine API, then export results as CSV.

AI use:
Used for debugging and syntax help/ parameter verfication
"""

import json
import time
import ee


# --------------------
# Config (edit here)
# --------------------
GADM_ADM3_ASSET = "projects/project-goodnight-485705/assets/gadm41_CHN_3"

FILTERS = {
    "COUNTRY": "China",
    "NAME_1": "Sichuan",
    "NAME_2": "Chengdu",
}

YEAR = 2023

# NDVI (MODIS)
MODIS_COLLECTION = "MODIS/061/MOD13Q1"
MODIS_BAND = "NDVI"
MODIS_SCALE_FACTOR = 0.0001
MODIS_SCALE_M = 250

# VIIRS (nightlights)
VIIRS_COLLECTION = "NOAA/VIIRS/DNB/MONTHLY_V1/VCMCFG"
VIIRS_BAND = "avg_rad"

# Export
EXPORT_DESCRIPTION = f"Chengdu_NDVI_VIIRS_ADM3_{YEAR}"
EXPORT_FOLDER = "GEE_Exports"
EXPORT_FORMAT = "CSV"

TILE_SCALE = 4


def init_ee() -> None:
    """Initialize Earth Engine (prompts auth if needed)."""
    try:
        ee.Initialize()
    except Exception:
        ee.Authenticate()
        ee.Initialize()


def apply_filters(fc: ee.FeatureCollection, filters: dict) -> ee.FeatureCollection:
    out = fc
    for k, v in filters.items():
        out = out.filter(ee.Filter.eq(k, v))
    return out


def annual_ndvi(year: int) -> ee.Image:
    start = ee.Date.fromYMD(year, 1, 1)
    end = ee.Date.fromYMD(year + 1, 1, 1)

    col = (
        ee.ImageCollection(MODIS_COLLECTION)
        .filterDate(start, end)
        .select(MODIS_BAND)
        .map(lambda img: img.multiply(MODIS_SCALE_FACTOR).copyProperties(img, img.propertyNames()))
    )

    return col.mean().rename("ndvi_mean")


def annual_viirs(year: int) -> ee.Image:
    start = ee.Date.fromYMD(year, 1, 1)
    end = ee.Date.fromYMD(year + 1, 1, 1)

    col = (
        ee.ImageCollection(VIIRS_COLLECTION)
        .filterDate(start, end)
        .select(VIIRS_BAND)
    )

    return col.mean().rename("viirs_mean_avg_rad")


def add_labels(fc: ee.FeatureCollection) -> ee.FeatureCollection:
    def _f(feat: ee.Feature) -> ee.Feature:
        return (
            feat.set("country", FILTERS["COUNTRY"])
                .set("province", FILTERS["NAME_1"])
                .set("city", FILTERS["NAME_2"])
                .set("year", YEAR)
        )
    return fc.map(_f)


def reduce_to_admin_units(units: ee.FeatureCollection, ndvi_img: ee.Image, viirs_img: ee.Image) -> ee.FeatureCollection:
    img = ee.Image.cat([ndvi_img, viirs_img])

    reduced = img.reduceRegions(
        collection=units,
        reducer=ee.Reducer.mean(),
        scale=MODIS_SCALE_M,
        tileScale=TILE_SCALE,
    )

    keep = [
        "year", "country", "province", "city",
        "GID_0", "GID_1", "GID_2", "GID_3",
        "NAME_0", "NAME_1", "NAME_2", "NAME_3",
        "VARNAME_3", "TYPE_3", "ENGTYPE_3",
        "ndvi_mean", "viirs_mean_avg_rad",
    ]

    def _select(f: ee.Feature) -> ee.Feature:
        props = ee.Dictionary.fromLists(
            ee.List(keep),
            ee.List(keep).map(lambda k: f.get(ee.String(k)))
        )
        return ee.Feature(None, props)

    return reduced.map(_select)


def export_to_drive(fc: ee.FeatureCollection) -> ee.batch.Task:
    task = ee.batch.Export.table.toDrive(
        collection=fc,
        description=EXPORT_DESCRIPTION,
        folder=EXPORT_FOLDER,
        fileFormat=EXPORT_FORMAT,
    )
    task.start()
    return task


def wait_for_task(task: ee.batch.Task, poll_seconds: int = 10, timeout_minutes: int = 60) -> None:
    deadline = time.time() + timeout_minutes * 60
    last_state = None

    while True:
        status = task.status()
        state = status.get("state", "UNKNOWN")

        if state != last_state:
            print(json.dumps(status, indent=2))
            last_state = state

        if state in {"COMPLETED", "FAILED", "CANCELLED"}:
            if state != "COMPLETED":
                raise RuntimeError(f"Export ended in state={state}. Status:\n{json.dumps(status, indent=2)}")
            print("Export completed.")
            return

        if time.time() > deadline:
            raise TimeoutError("Timed out waiting for export. Check the GEE Tasks tab / Google Drive.")

        time.sleep(poll_seconds)


def main() -> None:
    init_ee()

    adm3 = ee.FeatureCollection(GADM_ADM3_ASSET)
    chengdu = apply_filters(adm3, FILTERS)

    n = chengdu.size().getInfo()
    if n == 0:
        raise ValueError("No features matched the Chengdu filters. Check field names/values in the ADM3 asset.")
    print(f"Matched ADM3 units: {n}")

    chengdu = add_labels(chengdu)

    ndvi = annual_ndvi(YEAR)
    viirs = annual_viirs(YEAR)

    out = reduce_to_admin_units(chengdu, ndvi, viirs)

    # quick sanity sample
    print("Sample rows:")
    print(json.dumps(out.limit(3).getInfo(), indent=2))

    task = export_to_drive(out)
    print(f"Started export task: {task.id}")
    wait_for_task(task, poll_seconds=10, timeout_minutes=60)


if __name__ == "__main__":
    main()


Matched ADM3 units: 13
Sample rows:
{
  "type": "FeatureCollection",
  "columns": {},
  "features": [
    {
      "type": "Feature",
      "geometry": null,
      "id": "0000000000000000075b",
      "properties": {
        "ENGTYPE_3": "District",
        "GID_0": "CHN",
        "GID_1": "CHN.26_1",
        "GID_2": "CHN.26.2_1",
        "GID_3": "CHN.26.2.1_1",
        "NAME_0": null,
        "NAME_1": "Sichuan",
        "NAME_2": "Chengdu",
        "NAME_3": "Chengdu Q\u016b",
        "TYPE_3": "Sh\u00ecxi\u00e1q\u016b",
        "VARNAME_3": "Ch\u00e9ngd\u016b",
        "city": "Chengdu",
        "country": "China",
        "ndvi_mean": 0.4352977475213449,
        "province": "Sichuan",
        "viirs_mean_avg_rad": 14.569537491507962,
        "year": 2023
      }
    },
    {
      "type": "Feature",
      "geometry": null,
      "id": "00000000000000000765",
      "properties": {
        "ENGTYPE_3": "District",
        "GID_0": "CHN",
        "GID_1": "CHN.26_1",
        "GID_2": 

In [ ]:
# ---- parameters ----
# Change these values to run the same pipeline for a different city or year

GADM_ADM3_ASSET = "projects/project-goodnight-485705/assets/gadm41_CHN_3"

FILTERS = {
    "COUNTRY": "China",   # country name as listed in the boundary asset
    "NAME_1": "Sichuan",  # province
    "NAME_2": "Chengdu",  # city
}

YEAR = 2023  # analysis year


In [ ]:
# Initialize Google Earth Engine session
# Will prompt authentication if not already logged in
ee.Initialize()


In [ ]:
# Select Chengdu administrative units from the national ADM3 boundary file
chengdu = adm3.filter(ee.Filter.eq("NAME_2", "Chengdu"))


In [ ]:
# Build annual mean NDVI image from MODIS
# NDVI values are scaled to standard units


In [ ]:
# # # ^Apply scale factor to convert raw NDVI values


In [ ]:
# Aggregate values to admin boundaries
# This matches the geographic level of the survey data


In [ ]:
# Compute mean values within each district/county
